# Example cosmology forecasting with n(z)

This will use TQ Zhang's fisherA2Z code to do cosmological forecast from the nz estimates and realizations provided.

#### Standard imports

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.interpolate import interp1d

from fisherA2Z.fisher_flex import FisherFlex
from fisherA2Z import nz_decomposition as nzd
import tables_io
from scipy.interpolate import interp1d
import qp
from nz_data_challenge import forecast, utils
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

#### Input paths and constants

In [ ]:
submission_name = 'example'
test_only = False
if test_only:
    submit_dir = f'../submission_test/{submission_name}'
    test_suffix = 'ddf_00'
    truth_dir = '../public'
else:
    submit_dir = f'../submission/{submission_name}'
    test_suffix = 'wfd'    
    truth_dir = '../reserved'

taskset = 'taskset_1'
sim = 'cardinal'
scenario = '1yr'

truth_file = f'{truth_dir}/nz_challenge_{taskset}_{sim}_{scenario}_{test_suffix}.hdf5'
nz_estimates_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_{test_suffix}.hdf5"
nz_samples_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_samples_{test_suffix}.hdf5"
bin_assignments_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_{test_suffix}.hdf5"


#### Open the flies and read the data

In [ ]:
qp_nz_central = qp.read(nz_estimates_file)
qp_nz_samples = qp.read(nz_samples_file)
truth = tables_io.read(truth_file)
bhat_table = tables_io.read(bin_assignments_file)

In [ ]:
n_bins = qp_nz_samples.ancil['bin_idx'].max() + 1
n_samples = qp_nz_samples.ancil['i_realization'].max() + 1

#### Define the n(z) grid and compute pdfs

In [ ]:
z_grid = utils.Z_BIN_EDGES[taskset]
nz_bins = len(z_grid)

nz_realizations = qp_nz_samples.pdf(z_grid).reshape(n_bins,n_samples,nz_bins)
nz_central = qp_nz_central.pdf(z_grid)
true_redshifts = truth['redshift']
true_nz_distributions = utils.get_true_nz_distributions(
    true_redshifts,
    np.squeeze(bhat_table['tomo_bin_index']),
    z_grid,
    n_bins,
)


#### Plot stuff as a sanity check

In [ ]:
for i in range(n_bins):
    plt.xlabel('redshift z')
    plt.ylabel('nz central')
    for j in range(min(10, n_samples)):
       plt.plot(z_grid, nz_realizations[i, j, :], alpha = 0.1, color = 'black')
    plt.plot(z_grid, nz_central[i])

#### Grab the number of objects and compute the effective number density

In [ ]:
counts = np.squeeze(qp_nz_central.ancil['n_objects'])
neff = forecast.tomo_bins_effective_density(counts, taskset, sim, scenario)

print(f'effective number density in bins = {neff}')
print(f'number counts in bins = {counts}')

#### Start with a cosmic shear analysis

In [ ]:
res_y1_cs = forecast.fisher_forecast(
    qp_nz_central,
    qp_nz_samples,
    bhat_table,
    neff,
    mode='cosmic_shear',
)
s8, s8_err = res_y1_cs.s8()

print(f"LSST Y1 cs S_8 = {s8:.4f} +/- {s8_err:.4f}")

print(f"FoM(w0,wa)={res_y1_cs.fom('w_0', 'w_a'):8.2f}")


In [ ]:
params = ["omega_m", "sigma_8", "w_0", "w_a"]
fig = res_y1_cs.corner(params, color="C0", label="cosmic shear")

# fig = res_cut.corner(params, color="C3", label="3x2pt, $\\ell_{max}^{cs}=1500$", fig=fig)
fig.legend(loc="upper right", fontsize=10)
plt.show()

In [ ]:
res_y1_cs, bias_res_y1_cs = forecast.fisher_bias_forecast(
    qp_nz_central,
    bhat_table,
    neff,
    mode='cosmic_shear',
    truth=truth,
)
s8, s8_err = res_y1_cs.s8()

print(f"LSST Y1 cs S_8 = {s8:.4f} +/- {s8_err:.4f}")

print(f"FoM(w0,wa)={res_y1_cs.fom('w_0', 'w_a'):8.2f}")


#### Make a corner plot

In [ ]:
params = ["omega_m", "sigma_8", "w_0", "w_a"]
fig = res_y1_cs.corner(params, color="C0", label="cosmic shear")
bias_res_y1_cs.corner_arrows(params, fig=fig, shifted_contour=True, color="C1")

# fig = res_cut.corner(params, color="C3", label="3x2pt, $\\ell_{max}^{cs}=1500$", fig=fig)
fig.legend(loc="upper right", fontsize=10)
plt.show()

#### Now do 3x2 pt

In [ ]:
res_y1_3x2pt  = forecast.fisher_forecast(
    qp_nz_central,
    qp_nz_samples,
    bhat_table,
    neff,
    mode='3x2pt',
)


s8, s8_err = res_y1_3x2pt.s8()

print(f"LSST Y1 3x2pt S_8 = {s8:.4f} +/- {s8_err:.4f}")

print(f"FoM(w0,wa)={res_y1_cs.fom('w_0', 'w_a'):8.2f}")


#### Make a corner plot

In [ ]:
params = ["omega_m", "sigma_8", "w_0", "w_a"]

fig_3x2pt = res_y1_3x2pt.corner(params, color="C3", label="3x2pt")
fig_3x2pt.legend(loc="upper right", fontsize=10)


plt.show()

In [ ]:
res_y1_3x2pt, bias_res_y1_3x2pt  = forecast.fisher_bias_forecast(
    qp_nz_central,
    bhat_table,
    neff,
    mode='3x2pt',
    truth=truth,
)


s8, s8_err = res_y1_3x2pt.s8()

print(f"LSST Y1 3x2pt S_8 = {s8:.4f} +/- {s8_err:.4f}")

print(f"FoM(w0,wa)={res_y1_cs.fom('w_0', 'w_a'):8.2f}")


In [ ]:
params = ["omega_m", "sigma_8", "w_0", "w_a"]

fig_3x2pt = res_y1_3x2pt.corner(params, color="C3", label="3x2pt")
fig_3x2pt.legend(loc="upper right", fontsize=10)
bias_res_y1_3x2pt.corner_arrows(params, fig=fig_3x2pt, shifted_contour=True, color="C1")


plt.show()